# 🚀 Fundamentos de LLM para Developers & QA

Notebook diseñado para una exposición técnica clara, práctica y fácil de seguir.

**Base técnica usada desde tu `pyproject.toml`:**
- `openai>=1.109.1`
- `tiktoken>=0.11.0`
- `plotly>=6.3.0`
- `numpy>=2.3.3`
- `litellm>=1.77.5`

> Objetivo: entender cómo funciona un LLM por dentro, cómo se controla, y cómo explicarlo de forma simple frente a un equipo técnico.


## 🧠 Introducción

### 1) Frontier models
Son los modelos más avanzados disponibles hoy.  
Están en el **límite actual** de lo que la IA puede hacer en lenguaje, razonamiento, herramientas, visión y generación de código.

Ejemplos conocidos:
- GPT
- Claude
- Gemini

---

### 2) Open Source Models
Son modelos que puedes descargar, ejecutar localmente o adaptar en tu infraestructura.

Ejemplos:
- **LLaMA**
- **Mistral**
- **DeepSeek**
- **Ollama** *(Ollama no es un modelo: es una herramienta para correr modelos localmente)*

---

### 3) ¿Qué son los transformers?
Los LLM modernos están construidos sobre la arquitectura **Transformer**.

Idea simple:
- el modelo recibe un texto
- analiza la relación entre palabras
- decide qué partes del texto son más importantes
- predice la siguiente salida

La pieza clave es **attention**:
> el modelo "pone atención" a distintas partes del texto para entender mejor el contexto.

---

### 4) ¿Qué son los parámetros?
Los parámetros son los **pesos internos** del modelo.

Piensa en ellos como los números que el modelo ajustó durante el entrenamiento para aprender patrones del lenguaje.

En términos simples:
- más parámetros → más capacidad para representar patrones complejos
- pero más parámetros **no siempre** significa mejor respuesta para todos los casos

---

## Librerías que vamos a usar en este notebook
Usaremos herramientas que ya están declaradas en tu proyecto:
- `numpy` para simulaciones numéricas
- `plotly` para visualizaciones
- `tiktoken` para tokens
- `openai` para ejemplos de API
- `litellm` para ejemplo de capa unificada y costos


In [1]:
import numpy as np
import plotly.graph_objects as go

print("Notebook cargado correctamente ✅")


Notebook cargado correctamente ✅


## Celda 1 — ¿Qué es OpenAI?

**OpenAI es una empresa.**

No es ChatGPT y no es un modelo en sí mismo.

### Diferencia importante
- **OpenAI** → la empresa
- **GPT** → la familia de modelos
- **ChatGPT** → el producto o aplicación que usa esos modelos

### ¿Por qué importa entender esto?
Porque como ingenieros necesitamos separar:
- la empresa que construye la tecnología
- el modelo que procesa lenguaje
- la aplicación final que usa ese modelo

### Idea simple
> Empresa ≠ modelo ≠ producto


## Celda 2 — ¿Qué es un LLM realmente?

**LLM** significa **Large Language Model**.

En palabras simples:
> es un modelo entrenado con muchísimo texto para predecir la siguiente palabra o fragmento más probable.

### Lo más importante
Un LLM:
- no "piensa" como humano
- no "entiende" como humano
- no "recuerda" como humano

Lo que sí hace muy bien es:
- detectar patrones del lenguaje
- calcular probabilidades
- generar una continuación probable

### Ejemplo simple
Si le das:

`El cielo es de color ...`

el modelo evalúa varias palabras posibles:
- azul
- gris
- verde
- rojo

Y asigna probabilidades.  
Luego selecciona una según su estrategia de muestreo.

### Idea clave
> Un LLM no adivina por magia. Calcula probabilidades.


In [2]:
# Visualización simple: cómo un modelo "considera" varias siguientes palabras

words = ["azul", "gris", "rojo", "verde", "negro"]
probs = np.array([0.62, 0.18, 0.08, 0.07, 0.05], dtype=float)
probs = probs / probs.sum()

fig = go.Figure(
    data=[go.Bar(x=words, y=probs, text=[f"{p:.0%}" for p in probs], textposition="outside")]
)

fig.update_layout(
    title="Siguiente palabra probable para: 'El cielo es de color...'",
    xaxis_title="Palabra candidata",
    yaxis_title="Probabilidad",
    yaxis=dict(range=[0, 0.75]),
)

fig.show()

print("Palabra más probable:", words[int(np.argmax(probs))])


Palabra más probable: azul


## Celda 3 — ¿Qué son los tokens?

Un **token** es una unidad pequeña de texto que el modelo realmente procesa.

Un token puede ser:
- una palabra completa
- parte de una palabra
- signos de puntuación
- espacios o fragmentos especiales

### Idea importante
Los modelos **no leen texto como nosotros**.  
Lo convierten en tokens antes de procesarlo.

### ¿Por qué importa?
Porque:
- el costo normalmente se calcula por tokens
- el contexto tiene límite de tokens
- el rendimiento cambia según cuántos tokens envías y recibes

### Tokenizer oficial de OpenAI
Puedes ver cómo se divide un texto en tokens aquí:

**https://platform.openai.com/tokenizer**


In [3]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4")

texts = [
    "Hola, estoy aprendiendo LLMs.",
    "Tokenizar no es lo mismo que contar palabras.",
    "Developers y QA necesitan entender costo y contexto."
]

for text in texts:
    token_ids = encoding.encode(text)
    print("=" * 80)
    print("Texto:")
    print(text)
    print("\nToken IDs:")
    print(token_ids)
    print("Cantidad de tokens:", len(token_ids))


Texto:
Hola, estoy aprendiendo LLMs.

Token IDs:
[69112, 11, 82384, 68446, 37116, 445, 11237, 82, 13]
Cantidad de tokens: 9
Texto:
Tokenizar no es lo mismo que contar palabras.

Token IDs:
[3404, 11403, 912, 1560, 781, 36994, 1744, 89805, 74304, 13]
Cantidad de tokens: 10
Texto:
Developers y QA necesitan entender costo y contexto.

Token IDs:
[21076, 388, 379, 67008, 25203, 13145, 96537, 78949, 379, 77843, 13]
Cantidad de tokens: 11


## Celda 4 — ¿Qué es el contexto?

El **contexto** es toda la información que el modelo puede ver en una interacción.

Puede incluir:
- instrucciones del sistema
- mensajes del usuario
- respuestas anteriores
- documentos inyectados
- herramientas o resultados previos

### Limitaciones
El contexto **no es infinito**.

Si metes demasiado:
- sube el costo
- puede bajar la calidad
- puedes meter ruido irrelevante
- el modelo puede perder foco

### ¿Por qué es importante administrarlo?
Porque un buen sistema con IA no solo "manda texto":
- decide **qué sí enviar**
- decide **qué no enviar**
- resume historial
- recorta ruido
- prioriza contexto útil

### Idea clave
> Un usuario conversa. Un ingeniero administra contexto.

### Ilusión de memoria
Muchas personas creen que la IA “recuerda”.  
En realidad, muchas veces solo parece recordar porque tú le sigues mandando el historial.


In [4]:
# Ejemplo simple de "ilusión de memoria"

conversation_with_history = [
    {"role": "user", "content": "Mi nombre es Rager."},
    {"role": "assistant", "content": "Entendido, tu nombre es Rager."},
    {"role": "user", "content": "¿Cuál es mi nombre?"}
]

conversation_without_history = [
    {"role": "user", "content": "¿Cuál es mi nombre?"}
]

print("Conversación CON historial:")
for msg in conversation_with_history:
    print(f"- {msg['role']}: {msg['content']}")

print("\nConversación SIN historial:")
for msg in conversation_without_history:
    print(f"- {msg['role']}: {msg['content']}")

print("\nConclusión:")
print("Si no mandas el historial, el modelo no tiene por qué saber el nombre.")


Conversación CON historial:
- user: Mi nombre es Rager.
- assistant: Entendido, tu nombre es Rager.
- user: ¿Cuál es mi nombre?

Conversación SIN historial:
- user: ¿Cuál es mi nombre?

Conclusión:
Si no mandas el historial, el modelo no tiene por qué saber el nombre.


## Celda 5 — ¿Qué es un prompt?

Un **prompt** es la instrucción que le das al modelo.

No es solo una pregunta.  
Es la forma en la que defines:
- qué quieres
- cómo lo quieres
- con qué nivel de detalle
- en qué formato

### Ejemplo de prompt débil
`Explícame IA`

### Ejemplo de prompt mejor diseñado
`Explícame qué es un LLM en 3 puntos, nivel principiante, con un ejemplo pequeño y lenguaje simple.`

### Diferencia práctica
Un prompt mejor:
- reduce ambigüedad
- mejora claridad
- mejora formato
- mejora consistencia

### Idea clave
> Un prompt bien diseñado guía mejor al modelo.


In [ ]:
prompt_debil = "Explícame IA"

prompt_fuerte = (
    "Explícame qué es un LLM en 3 puntos, "
    "nivel principiante, con lenguaje simple y un ejemplo práctico."
)

print("Prompt débil:")
print(prompt_debil)
print("\nPrompt fuerte:")
print(prompt_fuerte)


## Celda 6 — ¿Qué son los roles? (`system`, `user`, `assistant`)

Cuando trabajas con un LLM en una conversación estructurada, normalmente usas roles.

### 1) `system`
Define reglas, tono, comportamiento y restricciones del modelo.

Ejemplos:
- “Eres un profesor claro y conciso”
- “Responde solo en español”
- “No inventes información”

### 2) `user`
Es lo que la persona pide o pregunta.

### 3) `assistant`
Es la respuesta del modelo.

### ¿Para qué sirve cada uno?
- `system` → controla el comportamiento general
- `user` → plantea la necesidad
- `assistant` → entrega la salida

### Idea clave
> El rol más poderoso es `system`, porque condiciona cómo se comportará el modelo.


In [5]:
# Ejemplo estructurado de roles

messages = [
    {"role": "system", "content": "Eres un arquitecto de software. Responde claro y en español."},
    {"role": "user", "content": "Explícame qué es un LLM en 2 puntos."},
    {"role": "assistant", "content": "Un LLM es un modelo de lenguaje entrenado con mucho texto."},
    {"role": "user", "content": "Ahora compáralo con una API tradicional."}
]

for msg in messages:
    print(f"[{msg['role'].upper()}]")
    print(msg["content"])
    print()


[SYSTEM]
Eres un arquitecto de software. Responde claro y en español.

[USER]
Explícame qué es un LLM en 2 puntos.

[ASSISTANT]
Un LLM es un modelo de lenguaje entrenado con mucho texto.

[USER]
Ahora compáralo con una API tradicional.



## Celda 7 — ¿Qué es `temperature`?

`temperature` controla qué tan **creativa o variada** será la salida.

### Intuición simple
- **temperature baja** → más predecible, más estable, más conservadora
- **temperature alta** → más variación, más creatividad, más riesgo de rarezas

### Cuándo usar cada una
- **baja**: clasificación, resúmenes, respuestas estables, producción
- **alta**: ideas, slogans, brainstorming, escritura creativa

### Idea clave
> `temperature` afecta cómo se reparte la probabilidad entre las opciones.


In [6]:
# Simulación local del efecto de temperature sobre probabilidades

base_words = ["azul", "gris", "rojo", "verde", "morado"]
base_probs = np.array([0.60, 0.20, 0.08, 0.07, 0.05], dtype=float)

def apply_temperature(probabilities, temperature: float):
    probabilities = np.array(probabilities, dtype=float)
    logits = np.log(probabilities + 1e-12)
    adjusted = np.exp(logits / temperature)
    adjusted = adjusted / adjusted.sum()
    return adjusted

temp_low = apply_temperature(base_probs, 0.4)
temp_high = apply_temperature(base_probs, 1.5)

fig_low = go.Figure(data=[go.Bar(x=base_words, y=temp_low, text=[f"{p:.0%}" for p in temp_low], textposition="outside")])
fig_low.update_layout(title="Temperature baja (0.4): distribución más concentrada", yaxis=dict(range=[0, 1]))
fig_low.show()

fig_high = go.Figure(data=[go.Bar(x=base_words, y=temp_high, text=[f"{p:.0%}" for p in temp_high], textposition="outside")])
fig_high.update_layout(title="Temperature alta (1.5): distribución más abierta", yaxis=dict(range=[0, 1]))
fig_high.show()


## Celda 8 — ¿Qué es `top_p`?

`top_p` es otra forma de controlar la generación.

En lugar de abrir o cerrar la distribución completa, hace esto:
- ordena las palabras por probabilidad
- toma solo las más probables hasta cubrir cierto porcentaje acumulado
- ignora el resto

### Ejemplo conceptual
Si `top_p = 0.80`, el modelo podría quedarse solo con las palabras más probables cuya suma alcance 80%.

### Interpretación simple
- `top_p` bajo → menos opciones, salida más controlada
- `top_p` alto → más opciones, salida más abierta

### Idea clave
> `top_p` controla cuántas opciones entran al juego.


In [8]:
# Simulación local de top_p

candidate_words = ["azul", "gris", "rojo", "verde", "morado", "blanco"]
candidate_probs = np.array([0.50, 0.20, 0.12, 0.08, 0.06, 0.04], dtype=float)

def top_p_filter(words, probabilities, top_p=0.8):
    pairs = sorted(zip(words, probabilities), key=lambda x: x[1], reverse=True)
    selected_words = []
    selected_probs = []
    cumulative = 0.0
    for word, prob in pairs:
        selected_words.append(word)
        selected_probs.append(prob)
        cumulative += prob
        if cumulative >= top_p:
            break
    selected_probs = np.array(selected_probs, dtype=float)
    selected_probs = selected_probs / selected_probs.sum()
    return selected_words, selected_probs

selected_words, selected_probs = top_p_filter(candidate_words, candidate_probs, top_p=0.80)

print("Palabras originales:", candidate_words)
print("Probabilidades originales:", candidate_probs)
print("\nPalabras seleccionadas con top_p=0.80:", selected_words)
print("Probabilidades re-normalizadas:", selected_probs)

fig = go.Figure(data=[go.Bar(x=selected_words, y=selected_probs, text=[f"{p:.0%}" for p in selected_probs], textposition="outside")])
fig.update_layout(title="Opciones consideradas después de aplicar top_p=0.80", yaxis=dict(range=[0, 1]))
fig.show()


Palabras originales: ['azul', 'gris', 'rojo', 'verde', 'morado', 'blanco']
Probabilidades originales: [0.5  0.2  0.12 0.08 0.06 0.04]

Palabras seleccionadas con top_p=0.80: ['azul', 'gris', 'rojo']
Probabilidades re-normalizadas: [0.6097561  0.24390244 0.14634146]


## Celda 9 — ¿Qué son los `max tokens`?

`max tokens` o `max_output_tokens` controla el **máximo de tokens que el modelo puede generar en la respuesta**.

### ¿Por qué importa?
Porque impacta:
- costo
- longitud de salida
- latencia
- control del formato

### Ejemplo práctico
Si pides:
- un resumen corto → conviene limitar salida
- una explicación detallada → necesitas más tokens

### Idea clave
> No siempre quieres la respuesta más larga. Quieres la respuesta adecuada.


In [9]:
# Simulación conceptual de max_output_tokens

examples = [
    {"input": "Resume la arquitectura en una frase.", "max_output_tokens": 20},
    {"input": "Explica la arquitectura en detalle.", "max_output_tokens": 150},
]

for ex in examples:
    print("=" * 80)
    print("Prompt:", ex["input"])
    print("max_output_tokens:", ex["max_output_tokens"])
    print("Interpretación:",
          "respuesta breve y controlada" if ex["max_output_tokens"] <= 30 else "respuesta más amplia y detallada")


Prompt: Resume la arquitectura en una frase.
max_output_tokens: 20
Interpretación: respuesta breve y controlada
Prompt: Explica la arquitectura en detalle.
max_output_tokens: 150
Interpretación: respuesta más amplia y detallada


## Celda 10 — ¿Qué es el determinismo en modelos?

El determinismo busca que una misma entrada produzca una salida igual o muy parecida de forma consistente.

### En la práctica
Mientras más aleatoriedad permites:
- más variación habrá

Mientras más control apliques:
- más consistente será la respuesta

### Cómo acercarte al determinismo
- `temperature` baja
- `top_p` bajo o controlado
- prompts estables
- mismo modelo
- mismo contexto
- si el proveedor lo soporta, usar `seed`

### Idea clave
> Producción normalmente necesita consistencia más que creatividad.


In [ ]:
# Simulación local de muestreo con y sin seed para ilustrar determinismo

rng_words = ["azul", "gris", "rojo", "verde"]
rng_probs = np.array([0.65, 0.20, 0.10, 0.05], dtype=float)

def sample_next_word(words, probs, seed=None):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(words), p=probs)
    return words[idx]

print("Sin seed (puede variar):")
for _ in range(5):
    print(sample_next_word(rng_words, rng_probs))

print("\nCon seed fija (misma salida en esta simulación):")
for _ in range(5):
    print(sample_next_word(rng_words, rng_probs, seed=42))


## Celda 11 — ¿Qué es LiteLLM?

**LiteLLM** es una librería open source que te da una **interfaz unificada** para llamar a muchos proveedores de LLM usando un formato similar.

### ¿Para qué sirve?
En vez de escribir integración distinta para cada proveedor, LiteLLM te ayuda a:
- unificar llamadas
- cambiar de proveedor más fácil
- centralizar costos y usage
- simplificar pruebas y migraciones

### Idea simple
> Es como una capa de abstracción para hablar con varios LLMs de forma parecida.

### Además
LiteLLM también puede ayudarte a ver:
- uso de tokens
- costo aproximado de una llamada
- compatibilidad entre proveedores

En el ejemplo siguiente usamos una respuesta simulada (`mock_response`) para no depender de una API key durante la exposición.


In [10]:
from litellm import completion

response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explícame qué es un LLM en una frase."}],
    mock_response="Un LLM es un modelo que predice texto basándose en patrones aprendidos."
)

print("Respuesta simulada:")
print(response.choices[0].message.content)

print("\nUso reportado:")
print(response.usage)

print("\nCosto estimado por LiteLLM (si está disponible):")
print(response._hidden_params.get("response_cost", 0))


Respuesta simulada:
Un LLM es un modelo que predice texto basándose en patrones aprendidos.

Uso reportado:
Usage(completion_tokens=20, prompt_tokens=10, total_tokens=30, completion_tokens_details=None, prompt_tokens_details=None)

Costo estimado por LiteLLM (si está disponible):
1.35e-05


## ✅ Conclusión

Si te llevas solo una idea de este notebook, que sea esta:

- un LLM trabaja con **tokens**
- decide por **probabilidades**
- depende del **contexto**
- obedece mejor con un buen **prompt**
- cambia de comportamiento con **roles** y parámetros como:
  - `temperature`
  - `top_p`
  - `max_output_tokens`

Y cuando quieres llevar esto a producción:
- buscas más control
- más consistencia
- mejor costo
- mejor observabilidad

Eso es lo que separa usar IA como usuario… de diseñarla como ingeniero.
